In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

f = 'melb_data.csv'
df_original = pd.read_csv(f)
df_original.loc[(df_original.Type == 'u') & (df_original.Landsize.isnull()), 'Landsize'] = 0
df_original["Car"] = df_original["Car"].fillna(0)
target_col = "Price"
feature_cols = ["Rooms", "Bedroom2", "Bathroom", "Car", "Landsize", "BuildingArea",
                "YearBuilt", "Distance", "Lattitude", "Longtitude", "Propertycount"]

df_base = df_original[feature_cols + [target_col]].copy()
df = df_base.dropna().copy()

df["PriceClass"] = pd.qcut(df[target_col], q=4, labels=False).astype(int)

X = df[feature_cols]
y = df["PriceClass"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]
n_classes = y_train.nunique()

print("Number of features:", n_features)
print("Number of classes:", n_classes)

X_train_tensor = torch.from_numpy(X_train_scaled.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test_scaled.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.to_numpy().astype(np.int64))
y_test_tensor = torch.from_numpy(y_test.to_numpy().astype(np.int64))

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.out = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.out(x)
        return x

model = MLP(input_dim=n_features, num_classes=n_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f}")

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.numpy())
        all_targets.extend(y_batch.numpy())

test_acc = accuracy_score(all_targets, all_preds)
print(f"\nTest accuracy (MLP, PyTorch): {test_acc:.4f}\n")

print("Classification report:")
print(classification_report(all_targets, all_preds))

Number of features: 11
Number of classes: 4
Epoch 1/30 - loss: 1.0380 - acc: 0.5399
Epoch 2/30 - loss: 0.7943 - acc: 0.6531
Epoch 3/30 - loss: 0.7505 - acc: 0.6770
Epoch 4/30 - loss: 0.7296 - acc: 0.6828
Epoch 5/30 - loss: 0.7158 - acc: 0.6921
Epoch 6/30 - loss: 0.7041 - acc: 0.6967
Epoch 7/30 - loss: 0.6953 - acc: 0.6941
Epoch 8/30 - loss: 0.6866 - acc: 0.7000
Epoch 9/30 - loss: 0.6823 - acc: 0.7020
Epoch 10/30 - loss: 0.6740 - acc: 0.7063
Epoch 11/30 - loss: 0.6713 - acc: 0.7085
Epoch 12/30 - loss: 0.6641 - acc: 0.7105
Epoch 13/30 - loss: 0.6591 - acc: 0.7133
Epoch 14/30 - loss: 0.6523 - acc: 0.7153
Epoch 15/30 - loss: 0.6522 - acc: 0.7160
Epoch 16/30 - loss: 0.6496 - acc: 0.7151
Epoch 17/30 - loss: 0.6439 - acc: 0.7158
Epoch 18/30 - loss: 0.6398 - acc: 0.7229
Epoch 19/30 - loss: 0.6393 - acc: 0.7193
Epoch 20/30 - loss: 0.6348 - acc: 0.7200
Epoch 21/30 - loss: 0.6311 - acc: 0.7269
Epoch 22/30 - loss: 0.6278 - acc: 0.7237
Epoch 23/30 - loss: 0.6274 - acc: 0.7229
Epoch 24/30 - loss: 0.

In [15]:
import torch.nn.functional as F
from torchvision import datasets, transforms
import random

torch.manual_seed(42)
random.seed(42)

transform = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])

train_dir = "train"
val_dir   = "valid"

train_set = datasets.ImageFolder(root=train_dir, transform=transform)
val_set   = datasets.ImageFolder(root=val_dir,   transform=transform)

classes = train_set.classes
num_classes = len(classes)

print("Number of classes:", num_classes)
print("Classes example:", classes[:10])
print("Train size:", len(train_set))
print("Val size:", len(val_set))

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=128, shuffle=False)

class ButterflyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32,  64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.drop = nn.Dropout(0.25)
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.bn4 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool2(x)
        x = self.drop(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.bn4(self.fc1(x)))
        x = self.drop(x)
        logits = self.fc2(x)
        return logits

model = ButterflyCNN(num_classes=num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = out.argmax(1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)

    avg_loss = total_loss / len(loader)
    acc = total_correct / total_samples
    return avg_loss, acc

def evaluate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            preds = out.argmax(1)
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

    avg_loss = total_loss / len(loader)
    acc = total_correct / total_samples
    return avg_loss, acc

n_epochs = 5

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate_one_epoch(model, val_loader, criterion)

    print(f"[{epoch+1:02d}] "
          f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in val_loader:
        out = model(x)
        preds = out.argmax(1)
        all_preds.extend(preds.numpy())
        all_targets.extend(y.numpy())

final_acc = accuracy_score(all_targets, all_preds)
print(f"\nFinal validation accuracy (ButterflyCNN): {final_acc:.4f}\n")

print("Classification report (first few classes):")
print(classification_report(all_targets, all_preds, target_names=classes, zero_division=0))

Number of classes: 100
Classes example: ['ADONIS', 'AFRICAN GIANT SWALLOWTAIL', 'AMERICAN SNOOT', 'AN 88', 'APPOLLO', 'ARCIGERA FLOWER MOTH', 'ATALA', 'ATLAS MOTH', 'BANDED ORANGE HELICONIAN', 'BANDED PEACOCK']
Train size: 12594
Val size: 500
[01] train_loss=2.8465 | train_acc=0.3541 | val_loss=1.6869 | val_acc=0.6180
[02] train_loss=1.4924 | train_acc=0.6445 | val_loss=1.2107 | val_acc=0.7060
[03] train_loss=0.9763 | train_acc=0.7565 | val_loss=1.0032 | val_acc=0.7620
[04] train_loss=0.6478 | train_acc=0.8416 | val_loss=0.9135 | val_acc=0.7720
[05] train_loss=0.4428 | train_acc=0.8914 | val_loss=0.8337 | val_acc=0.7740

Final validation accuracy (ButterflyCNN): 0.7740

Classification report (first few classes):
                           precision    recall  f1-score   support

                   ADONIS       0.40      0.80      0.53         5
AFRICAN GIANT SWALLOWTAIL       1.00      0.80      0.89         5
           AMERICAN SNOOT       0.80      0.80      0.80         5
         

In [19]:
import re
from collections import Counter

df = pd.read_csv("spam.csv", encoding="latin-1")
df = df.rename(columns={"v1": "label", "v2": "text"})
df = df[["text", "label"]].copy()
df = df.dropna()
df["label"] = df["label"].map({"ham": 0, "spam": 1})

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
texts = df["clean_text"].values
labels = df["label"].values

X_train_texts, X_test_texts, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

train_tokens = [t.split() for t in X_train_texts]
counter = Counter(tok for sent in train_tokens for tok in sent)

min_freq = 2

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)

PAD_IDX = vocab[PAD_TOKEN]
UNK_IDX = vocab[UNK_TOKEN]
vocab_size = len(vocab)

def encode_sentence(text, vocab, unk_idx=UNK_IDX):
    tokens = text.split()
    return [vocab.get(tok, unk_idx) for tok in tokens]

MAX_LEN = 40

def encode_and_pad(texts, vocab):
    encoded = []
    for txt in texts:
        ids = encode_sentence(txt, vocab)
        ids = ids[:MAX_LEN]
        if len(ids) < MAX_LEN:
            ids = ids + [PAD_IDX] * (MAX_LEN - len(ids))
        encoded.append(ids)
    return np.array(encoded, dtype="int64")

X_train_ids = encode_and_pad(X_train_texts, vocab)
X_test_ids = encode_and_pad(X_test_texts, vocab)

X_train_tensor = torch.tensor(X_train_ids, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_ids, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

class SpamBiLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_idx,
                 num_layers=1, bidirectional=True, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), 1)

    def forward(self, x):
        x = self.embedding(x)
        out, (h, c) = self.lstm(x)
        if self.lstm.bidirectional:
            h_fwd = h[-2, :, :]
            h_bwd = h[-1, :, :]
            h_cat = torch.cat([h_fwd, h_bwd], dim=1)
        else:
            h_cat = h[-1, :, :]
        logits = self.fc(h_cat)
        return logits.squeeze(1)

emb_dim = 100
hidden_dim = 128

model_a = SpamBiLSTM(vocab_size=vocab_size, emb_dim=emb_dim, hidden_dim=hidden_dim, pad_idx=PAD_IDX)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_a.parameters(), lr=1e-3)

n_epochs = 5

for epoch in range(n_epochs):
    model_a.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model_a(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        train_correct += (preds == yb.long()).sum().item()
        train_total += xb.size(0)
    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total

    model_a.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model_a(xb)
            loss = criterion(logits, yb)
            test_loss += loss.item() * xb.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            test_correct += (preds == yb.long()).sum().item()
            test_total += xb.size(0)
    avg_test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    print(f"[Random emb] Epoch {epoch+1}/{n_epochs} - "
          f"train_loss={avg_train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={avg_test_loss:.4f}, test_acc={test_acc:.4f}")

model_a.eval()
all_preds_a = []
all_targets_a = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model_a(xb)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        all_preds_a.extend(preds.numpy().astype(int))
        all_targets_a.extend(yb.numpy().astype(int))

print("\nClassification report (random embeddings):")
print(classification_report(all_targets_a, all_preds_a, target_names=["ham", "spam"], zero_division=0))

glove_path = "glove.6B.100d.txt"

embeddings_index = {}
with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.strip().split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = coefs

emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, emb_dim)).astype("float32")
emb_matrix[PAD_IDX] = np.zeros(emb_dim, dtype="float32")

found = 0
for word, idx in vocab.items():
    vec = embeddings_index.get(word)
    if vec is not None:
        emb_matrix[idx] = vec
        found += 1

print(f"GloVe vectors found for {found} words out of {vocab_size}")

pretrained_embedding = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=True, padding_idx=PAD_IDX)

model_b = SpamBiLSTM(vocab_size=vocab_size, emb_dim=emb_dim, hidden_dim=hidden_dim, pad_idx=PAD_IDX)

model_b.embedding = pretrained_embedding

criterion_b = nn.BCEWithLogitsLoss()
optimizer_b = torch.optim.Adam(model_b.parameters(), lr=1e-3)

n_epochs_b = 5

for epoch in range(n_epochs_b):
    model_b.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for xb, yb in train_loader:
        optimizer_b.zero_grad()
        logits = model_b(xb)
        loss = criterion_b(logits, yb)
        loss.backward()
        optimizer_b.step()
        train_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        train_correct += (preds == yb.long()).sum().item()
        train_total += xb.size(0)
    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total

    model_b.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = model_b(xb)
            loss = criterion_b(logits, yb)
            test_loss += loss.item() * xb.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            test_correct += (preds == yb.long()).sum().item()
            test_total += xb.size(0)
    avg_test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    print(f"[GloVe emb] Epoch {epoch+1}/{n_epochs_b} - "
          f"train_loss={avg_train_loss:.4f}, train_acc={train_acc:.4f}, "
          f"test_loss={avg_test_loss:.4f}, test_acc={test_acc:.4f}")

model_b.eval()
all_preds_b = []
all_targets_b = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model_b(xb)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        all_preds_b.extend(preds.numpy().astype(int))
        all_targets_b.extend(yb.numpy().astype(int))

print("\nClassification report (GloVe embeddings):")
print(classification_report(all_targets_b, all_preds_b, target_names=["ham", "spam"], zero_division=0))

[Random emb] Epoch 1/5 - train_loss=0.3324, train_acc=0.8775, test_loss=0.1934, test_acc=0.9220
[Random emb] Epoch 2/5 - train_loss=0.1333, train_acc=0.9580, test_loss=0.1024, test_acc=0.9650
[Random emb] Epoch 3/5 - train_loss=0.0638, train_acc=0.9821, test_loss=0.0810, test_acc=0.9767
[Random emb] Epoch 4/5 - train_loss=0.0396, train_acc=0.9888, test_loss=0.0876, test_acc=0.9713
[Random emb] Epoch 5/5 - train_loss=0.0276, train_acc=0.9924, test_loss=0.0812, test_acc=0.9812

Classification report (random embeddings):
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.96      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

GloVe vectors found for 3150 words out of 3424
[GloVe emb] Epoch 1/5 - train_loss=0.3570, train_acc=0.8735, test_loss=0.2432, test_acc=0.9022
[GloVe 